In [ ]:
import osimport randomimport shutilimport sysimport numpy as np

In [ ]:
import osimport sys_here = os.getcwd()for _c in (_here, os.path.join(_here, "notebooks", "test")):    if os.path.exists(os.path.join(_c, "nb_temp_setup.py")):        if _c not in sys.path:            sys.path.insert(0, _c)        breakelse:    raise FileNotFoundError(        "nb_temp_setup.py not found in the working directory or "        "notebooks/test. Run this notebook from notebooks/test or the repository root."    )from nb_temp_setup import cleanup_nb_temp_dir, setup_nb_temp_dir# All relative-path artifacts (project dir, SQLite DB, credentials, logs)# are now written to this per-run temp folder instead of the repository.nb_temp_dir, repo_root = setup_nb_temp_dir()print(f"Notebook output directory: {nb_temp_dir}")print(f"Repository root: {repo_root}")

In [ ]:
for dir_to_remove in ["obs_test_project"]:
    try:
        shutil.rmtree(dir_to_remove, ignore_errors=True)
    except Exception as e:
        msg = (
            f"Failed to clean up '{dir_to_remove}' directory: {e}. "
            "Critical error - cannot start with stale data."
        )
        raise RuntimeError(
            msg,
        ) from e

print("Previous outputs cleaned.")

In [ ]:
from pat2vec.util.docker_elastic import ElasticContainer

es_container = ElasticContainer()
es_container.stop()

print("Starting Elasticsearch container (this may take a few seconds)...")
if not es_container.start():
    msg = "Failed to start Elasticsearch container. Check if Docker is running."
    raise RuntimeError(
        msg,
    )

host, username, password = es_container.get_credentials()

creds_filename = "test_elastic_credentials.py"
creds_content = f"""
username = "{username}"
password = "{password}"
api_key = None
hosts = ["{host}"]
"""

with open(creds_filename, "w") as f:
    f.write(creds_content)

print(f"Created '{creds_filename}' pointing to test cluster at {host}")

In [ ]:
from pat2vec.util.config_pat2vec import config_classfrom pat2vec.util.get_dummy_data_cohort_searcher import populate_elastic_with_dummy_dataschema_path = os.path.abspath("test_files/elastic_schemas.json")

In [ ]:
print("Populating test Elasticsearch cluster with dummy data...")
patient_ids = populate_elastic_with_dummy_data(config_populate, n_patients=5)

print()
print("Population complete.")
print(f"Generated {len(patient_ids)} dummy patients.")
print(f"Patient IDs: {patient_ids}")

In [ ]:
from pat2vec.pat2vec_search.cogstack_search_methods import initialize_cogstack_client

cs = initialize_cogstack_client(config_populate)

indices = ["epr_documents", "basic_observations", "observations", "order", "pims_apps"]
print("Refreshing indices...")
cs.elastic.indices.refresh(index=indices, ignore_unavailable=True)
print("Indices refreshed.")

print()
print("Index Status:")
for index in indices:
    try:
        if cs.elastic.indices.exists(index=index):
            count = cs.elastic.count(index=index)["count"]
            print(f"  - {index:<20}: {count} documents")
        else:
            # Skip epr_documents if not available (may be empty)
            if index == "epr_documents":
                print(f"  - {index:<20}: skip (empty or not created)")
                continue
            # For other missing indices, still raise an error as expected by the test
            msg = f"Index not created: {index}"
            raise RuntimeError(msg)
    except Exception as e:
        msg = f"Error checking index {index}: {e}"
        raise RuntimeError(msg)

In [ ]:
PROJ_NAME = os.path.join(repo_root, "obs_test_project")DB_FILENAME = "temp_obs_db.sqlite"DB_PATH = os.path.join(PROJ_NAME, "outputs", DB_FILENAME)os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

In [ ]:
from pat2vec.util.logger_setup import setup_logger

logger = setup_logger()
print("Logger initialized.")

In [ ]:
from pat2vec.util.config_pat2vec import config_class

config_obj = config_class(
    proj_name=PROJ_NAME,
    credentials_path=creds_filename,
    current_path_dir="",
    main_options={"obs": True},
    batch_mode=True,
    verbosity=0,
    random_seed_val=random_seed_value,
    testing=True,
    testing_elastic=True,
    dummy_medcat_model=True,
    use_controls=False,
    medcat=False,
    start_time=None,
    patient_id_column_name="client_idcode",
    annot_filter_options={},
    shuffle_pat_list=False,
    storage_backend="database",
    check_patient_existence=False,
    db_connection_string=db_connection_string,
    treatment_doc_filename="test_files/treatment_docs.csv",
    all_patient_list=patient_ids,
)

print("pat2vec configuration created with obs-only sources and database backend.")

In [ ]:
from pat2vec.main_pat2vec import main

try:
    pat2vec_obj = main(
        cogstack=True,
        use_filter=False,
        json_filter_path=None,
        random_seed_val=random_seed_value,
        hostname=None,
        config_obj=config_obj,
    )
except FileNotFoundError as e:
    msg = f"Failed to initialize pipeline: config path invalid. Error details: {e}."
    raise RuntimeError(
        msg,
    ) from e
except ValueError as e:
    msg = f"Failed to initialize pipeline: invalid configuration. Error details: {e}."
    raise RuntimeError(
        msg,
    ) from e
except RuntimeError as e:
    msg = f"Failed to initialize pipeline: initialization failure. Error details: {e}."
    raise RuntimeError(
        msg,
    ) from e
except Exception as e:
    msg = f"Failed to initialize pipeline: unexpected error. Error details: {e}."
    raise RuntimeError(
        msg,
    ) from e

print("pat2vec object initialized.")
print(f"Patient list: {pat2vec_obj.all_patient_list}")

In [ ]:
if not pat2vec_obj.all_patient_list:
    msg = (
        "No patients in patient list after initialization. "
        "This indicates a critical failure in data loading or filtering."
    )
    raise RuntimeError(
        msg,
    )

print(f"Processing patient: {pat2vec_obj.all_patient_list[0]}")

try:
    pat2vec_obj.pat_maker(0)
except Exception as e:
    msg = (
        f"Failed to process patient 0 with pat_maker: {e}. "
        "Critical error - pipeline failed to extract features."
    )
    raise RuntimeError(
        msg,
    ) from e

print("Patient feature extraction complete.")

In [ ]:
from pat2vec.util.helper_functions import get_all_features

all_features = get_all_features(config_obj)

if all_features.empty:
    msg = (
        "FATAL ERROR: get_all_features returned an empty DataFrame. "
        "This indicates a critical failure in the pat2vec pipeline. "
        "No features were extracted or saved to the database."
    )
    raise RuntimeError(
        msg,
    )

print(f"Successfully retrieved {all_features.shape[0]} rows from database.")

In [ ]:
all_features_alt = pat2vec_obj.get_all_features()

if all_features_alt.empty:
    msg = (
        "FATAL ERROR: pat2vec_obj.get_all_features() returned an empty DataFrame. "
        "This indicates a critical failure in feature storage."
    )
    raise RuntimeError(
        msg,
    )

print(f"pat2vec_obj.get_all_features(): {all_features_alt.shape[0]} rows retrieved.")

In [ ]:
from pat2vec.util.post_processing import extract_datetime_to_column

df_with_datetime = extract_datetime_to_column(all_features)

print()
print("=== OUTPUT FEATURES DATAFRAME ===")
print(f"Shape: {df_with_datetime.shape}")
print(f"Total features: {len(df_with_datetime.columns)}")

if not df_with_datetime.empty:
    print()
    print("First 3 rows:")
    print(df_with_datetime.head(3))
else:
    msg = "DataFrame is empty after datetime extraction. Critical error - no features to extract."
    raise RuntimeError(
        msg,
    )

In [ ]:
print("\n=== DEMONSTRATING DATA RETRIEVAL FOR OBS MODE ===")

all_pat_list = pat2vec_obj.all_patient_list

# Use get_all_features to retrieve all features from database
obs_features = pat2vec_obj.get_all_features()

if obs_features.empty:
    msg = (
        "FATAL ERROR: get_all_features() returned empty DataFrame. "
        "This indicates a critical failure in feature storage."
    )
    raise RuntimeError(
        msg,
    )

print(f"Successfully retrieved {obs_features.shape[0]} feature rows from database.")
print(f"Total features: {len(obs_features.columns)}")

# Show columns that start with 'obs_' to demonstrate observation features
obs_cols = [c for c in obs_features.columns if "obs_" in c.lower()]
print(f"Observation-related features ({len(obs_cols)}):")
for col in sorted(obs_cols)[:10]:  # Show first 10
    print(f"  - {col}")
if len(obs_cols) > 10:
    print(f"  ... and {len(obs_cols) - 10} more")

# Display sample of observation features
print("\nSample observation feature data:")
sample_df = obs_features.head(3)
print(sample_df[obs_cols[:5]] if len(obs_cols) >= 5 else sample_df)

In [ ]:
try:    if os.path.exists(DB_PATH):        os.remove(DB_PATH)        print(f"Removed database: {DB_PATH}")except Exception as e:    raise RuntimeError(        f"Failed to remove database file '{DB_PATH}': {e}. Critical error - cleanup incomplete."    ) from etry:    if os.path.exists(PROJ_NAME):        shutil.rmtree(PROJ_NAME, ignore_errors=False)        print(f"Removed project directory: {PROJ_NAME}")except Exception as e:    raise RuntimeError(        f"Failed to remove '{PROJ_NAME}' directory: {e}. Critical error - cleanup incomplete."    ) from etry:    if os.path.exists(creds_filename):        os.remove(creds_filename)        print(f"Removed Elasticsearch credentials: {creds_filename}")except Exception as e:    raise RuntimeError(        f"Failed to remove Elasticsearch credentials file '{creds_filename}': {e}. "        "Critical error - cleanup incomplete."    ) from etry:    es_container.stop()    print("Stopped Elasticsearch test container.")except Exception as e:    print(f"Warning: could not stop Elasticsearch container: {e}")cleanup_nb_temp_dir(nb_temp_dir)print(f"Removed temp output directory: {nb_temp_dir}")

In [ ]:
assert not os.path.exists(DB_PATH), "Database file still exists!"assert not os.path.exists(PROJ_NAME), "Project directory still exists!"assert not os.path.exists(creds_filename), "Elasticsearch credentials file still exists!"print("All cleanup verified - no residual files remain.")print("\n=== TEST SUCCESSFUL ===")